In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from homonym import step1, step2, step3, step4, step5, fix1, evaluation, prompts

load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed1_03_homonymous"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy()

flow_all, flow_filtered = step1.run_step1(current_df)
if flow_filtered:
    item = flow_filtered[0]
    print(f"Sample of flow_filtered: {{'activity': '{item['activity']}', 'predecessors': {item['predecessors'][:1]}..., 'successors': {item['successors'][:1]}...}}")

    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

targets = list(homonym_candidates.keys())
print(f"\n>>> Homonym Candidates Filtered: {len(targets)} activities")
if targets:
    first_target = targets[0]
    combos = homonym_candidates[first_target]
    print(f"\nHomonym Label (Sample): {first_target}")
    for i, combo in enumerate(combos[:3]):
        print(f"  Clean Label Candidate {i+1}: {combo}")
    if len(combos) > 3:
        print(f"  ... and {len(combos) - 3} more candidates")
    if len(targets) > 1:
        print(f"\n... and {len(targets) - 1} more target activities found.")
else:
    print("No homonym candidates discovered.")

res_s3 = step3.run_step3(llm, 
                         MODEL, 
                         llm_repetition, 
                         homonym_candidates, 
                         flow_all, 
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP3, 
                         prompts.USER_PROMPT_HOMONYM_STEP3
                        )
print("--- Step 3 Results Sample ---")

target_keys = list(res_s3.keys())

for target in target_keys[:2]:
    candidates = res_s3[target]
    print(f"Target: [{target}]")
    print(f"  └─ Sample: {candidates[0]} ({len(candidates)} total candidates)")

if len(target_keys) > 2:
    print(f"\n... and {len(target_keys) - 2} more targets.")
    
res_s4 = step4.run_step4(llm,
                         MODEL,
                         llm_repetition,
                         res_s3,
                         current_df,
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1,
                         prompts.USER_PROMPT_HOMONYM_STEP4_1,
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2,
                         prompts.USER_PROMPT_HOMONYM_STEP4_2
                        )

print("--- Step 4 Results Sample ---")

target_keys = list(res_s4.keys())

for target in target_keys[:2]:
    candidates = res_s4[target]
    print(f"Target: [{target}]")
    print(f"  └─ Sample: {candidates[0]} ({len(candidates)} total candidates)")

if len(target_keys) > 2:
    print(f"\n... and {len(target_keys) - 2} more targets.")

res_s5 = step5.run_step5(
    llm, 
    MODEL,
    llm_repetition,
    res_s4, 
    current_df, 
    prompts.SYSTEM_PROMPT_HOMONYM_STEP5, 
    prompts.USER_PROMPT_HOMONYM_STEP5
)

print("------------------Step 5 Results------------------")
print(json.dumps(res_s5, indent=2, ensure_ascii=False))

df_homonym = df[df['label'].notna()].copy()
df_homonym['clean_activity'] = df_homonym['label'].str.extract(r'\((.*?)\)')

homonym_answer = (
    df_homonym.groupby('activity')['clean_activity']
    .unique()
    .apply(list)
    .to_dict()
)
print("------------------ANSWER------------------")
print(json.dumps(homonym_answer, indent=4, ensure_ascii=False))


>>> Running Step 1
    - Total activities: 13
    - Potential homonym candidates: 9
Sample of flow_filtered: {'activity': 'Check for completeness', 'predecessors': ['info received']..., 'successors': ['Request info']...}
>>> Running Step 2
  - Match found for 'Check for completeness': 7 combinations
  - Match found for 'Decision review': 5 combinations
  - Match found for 'Information exchange': 5 combinations
  - Match found for 'Perform checks': 4 combinations
  - Match found for 'Request info': 3 combinations
  - Match found for 'Verification process': 6 combinations
  - Match found for 'info received': 4 combinations
  - Match found for 'notify reject': 3 combinations

>>> Homonym Candidates Filtered: 8 activities

Homonym Label (Sample): Check for completeness
  Clean Label Candidate 1: ['Notify accept', 'info received']
  Clean Label Candidate 2: ['Notify accept', 'review request received']
  Clean Label Candidate 3: ['Request info', 'info received']
  ... and 4 more candidates



In [ ]:
LOG_NAME = "pub_seed1_03_homonymous"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy()

flow_all, flow_filtered = step1.run_step1(current_df)
if flow_filtered:
    item = flow_filtered[0]
    print(f"Sample of flow_filtered: {{'activity': '{item['activity']}', 'predecessors': {item['predecessors'][:1]}..., 'successors': {item['successors'][:1]}...}}")

    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

targets = list(homonym_candidates.keys())
print(f"\n>>> Homonym Candidates Filtered: {len(targets)} activities")
if targets:
    first_target = targets[0]
    combos = homonym_candidates[first_target]
    print(f"\nHomonym Label (Sample): {first_target}")
    for i, combo in enumerate(combos[:3]):
        print(f"  Clean Label Candidate {i+1}: {combo}")
    if len(combos) > 3:
        print(f"  ... and {len(combos) - 3} more candidates")
    if len(targets) > 1:
        print(f"\n... and {len(targets) - 1} more target activities found.")
else:
    print("No homonym candidates discovered.")

res_s3 = step3.run_step3(llm, 
                         MODEL, 
                         llm_repetition, 
                         homonym_candidates, 
                         flow_all, 
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP3, 
                         prompts.USER_PROMPT_HOMONYM_STEP3
                        )
print("--- Step 3 Results Sample ---")

target_keys = list(res_s3.keys())

for target in target_keys[:2]:
    candidates = res_s3[target]
    print(f"Target: [{target}]")
    print(f"  └─ Sample: {candidates[0]} ({len(candidates)} total candidates)")

if len(target_keys) > 2:
    print(f"\n... and {len(target_keys) - 2} more targets.")
    
res_s4 = step4.run_step4(llm,
                         MODEL,
                         llm_repetition,
                         res_s3,
                         current_df,
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1,
                         prompts.USER_PROMPT_HOMONYM_STEP4_1,
                         prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2,
                         prompts.USER_PROMPT_HOMONYM_STEP4_2
                        )

print("--- Step 4 Results Sample ---")

target_keys = list(res_s4.keys())

for target in target_keys[:2]:
    candidates = res_s4[target]
    print(f"Target: [{target}]")
    print(f"  └─ Sample: {candidates[0]} ({len(candidates)} total candidates)")

if len(target_keys) > 2:
    print(f"\n... and {len(target_keys) - 2} more targets.")

res_s5 = step5.run_step5(
    llm, 
    MODEL,
    llm_repetition,
    res_s4, 
    current_df, 
    prompts.SYSTEM_PROMPT_HOMONYM_STEP5, 
    prompts.USER_PROMPT_HOMONYM_STEP5
)

print("------------------Step 5 Results------------------")
print(json.dumps(res_s5, indent=2, ensure_ascii=False))

df_homonym = df[df['label'].notna()].copy()
df_homonym['clean_activity'] = df_homonym['label'].str.extract(r'\((.*?)\)')

homonym_answer = (
    df_homonym.groupby('activity')['clean_activity']
    .unique()
    .apply(list)
    .to_dict()
)
print("------------------ANSWER------------------")
print(json.dumps(homonym_answer, indent=4, ensure_ascii=False))
